# 2. Working with AI

**기본 경로는 자기 AI 도구로 요청하고 결과 파일을 실제로 완성하는 실습입니다.** 아래 대체 fixture는 AI 계정이나 도구 접근이 없을 때만 씁니다. 고정 fixture의 출력은 실제 모델 응답이나 실제 도구 호출 기록이 아닙니다.

Python 문법 구현은 요구하지 않습니다. 제공된 검사 셀은 그대로 실행하고 결과를 읽으면 됩니다.

## 기본 경로 A. 실제 AI로 문서 만들기

1. 사용 중인 AI 채팅을 열고 아래 source packet을 첨부하거나 붙여 넣으세요. D1과 D2를 구분하세요.
2. 아래 첫 요청을 보내 실제 응답을 받으세요.
3. 읽는 사람, 예시, 출력 형식으로 적어도 한 번 후속 수정하세요.
4. 최종 문서를 `inspection-brief.md`로 저장하고 다시 여세요.
5. 첫 요청, 후속 요청, 결과 파일, 확인한 내용을 남기세요.

**첫 요청**

> Create inspection-brief.md from D1, D2 and inspection.csv. Summarize the current rule, compare the guides, and include a compact result table and open questions. Use only supplied sources. Ask about facts that change the answer.

**후속 요청 예시**

> Replace the comparison prose with a D1/D2 table. Keep every raw row and warn about repeated IDs. Shorten repeated explanation while preserving the five counts.

Source packet:

```text
D1
D1: Earlier guide. Flag scores >= 0.60. Omit blank scores from the report. Keep row order. Do not delete source data.

D2
D2: Current guide. Flag valid scores >= 0.50. Keep every raw row. Mark blank as missing, scores outside [0,1] as invalid. Warn about repeated IDs. Do not delete source data. Report raw, valid, flagged, missing and invalid counts.

inspection.csv
id,score
A01,0.20
A02,0.50
A03,0.80
A04,
A05,1.20
A06,0.10
A06,0.60
A08,0.95
```

## 결과 확인

D2의 threshold는0.50이고 blank는 missing으로 남깁니다. 원본8행을 보존하고 A06 두 행을 모두 남겨야 합니다. 도구에서 실제로 만들고 저장한 파일을 열어 아래 독립 계산과 비교하세요.

In [1]:
import csv,io,math
from collections import Counter
source_csv='id,score\nA01,0.20\nA02,0.50\nA03,0.80\nA04,\nA05,1.20\nA06,0.10\nA06,0.60\nA08,0.95\n'
raw=list(csv.DictReader(io.StringIO(source_csv)))
valid=[r for r in raw if r["score"] and math.isfinite(float(r["score"])) and 0<=float(r["score"])<=1]
summary={"raw":len(raw),"valid":len(valid),"flagged":sum(float(r["score"])>=.5 for r in valid),
         "missing":sum(not r["score"] for r in raw),"invalid":len(raw)-len(valid)-sum(not r["score"] for r in raw)}
assert summary=={"raw":8,"valid":6,"flagged":4,"missing":1,"invalid":1}
print("Independent fixture counts:",summary)
print("Repeated IDs:",[k for k,v in Counter(r["id"]for r in raw).items()if v>1])

Independent fixture counts: {'raw': 8, 'valid': 6, 'flagged': 4, 'missing': 1, 'invalid': 1}
Repeated IDs: ['A06']


## 기본 경로의 과제

- 같은 자료로 shift supervisor용 짧은 문서와 engineer용 상세 문서를 AI에 요청하세요. 사실이 유지되는지 비교하세요.
- 새 대화에 원본과 짧은 handoff만 주고 A06에 관한 open question을 보강하세요. 실제 응답을 비교하세요.

모범 표현은 하나가 아닙니다. 산출물, 출처, 후속 수정의 관계가 보이면 됩니다.

## 대체 경로 B. 도구 접근이 없는 경우

다음 문서는 사람이 작성한 오류 fixture입니다. 이것을 AI 응답이라고 부르지 마세요. 현재 규칙, blank 처리, 결과 숫자를 고친 뒤 대체 경로임을 제출 기록에 남깁니다.

In [2]:
fixture_draft={"threshold":.60,"blank_policy":"drop","raw_rows":7,"valid":6,"flagged":3,"duplicate_warning":[]}
print("Supplied fixture draft, not a model response:",fixture_draft)
assert fixture_draft["threshold"]!=.50

Supplied fixture draft, not a model response: {'threshold': 0.6, 'blank_policy': 'drop', 'raw_rows': 7, 'valid': 6, 'flagged': 3, 'duplicate_warning': []}


### 대체 경로의 실행 가능한 해답

본문은 Current Rule / Changes / Results / Open Questions로 나눕니다. unknown은 새 사실로 채우지 않습니다.

In [3]:
reference={"threshold":.50,"blank_policy":"keep as missing","raw_rows":8,"valid":6,"flagged":4,"duplicate_warning":["A06"]}
assert reference["raw_rows"]==summary["raw"]
assert reference["flagged"]==summary["flagged"]
print("Human-authored reference for the alternative route:",reference)
print("Open question: Do the two A06 rows represent separate inspections?")

Human-authored reference for the alternative route: {'threshold': 0.5, 'blank_policy': 'keep as missing', 'raw_rows': 8, 'valid': 6, 'flagged': 4, 'duplicate_warning': ['A06']}
Open question: Do the two A06 rows represent separate inspections?


## 완료 기록

사용한 AI 도구와 실제 수행한 단계를 적으세요. 도구를 쓰지 못했다면 “대체 경로, 고정 fixture 사용”이라고 적고 실제 AI 호출을 했다고 쓰지 않습니다.